# Performance Tear Sheet

**Use after source analytics:** Reporting notebooks render results produced by the pricing, analytics, statement, and portfolio notebooks; they do not replace those calculation workflows.

**Purpose:** Render performance, drawdown, rolling, and distribution statistics for investment review.

**Prerequisites:** `03_analytics/performance_analytics.ipynb`.

**What you'll learn:**

- Prepare return/performance series.
- Render `reporting.performance_tearsheet`.
- Use a fixed generated date for reproducible notebooks.

A **performance tear sheet** is a single-page HTML report that summarises a return series: headline KPIs (total return, CAGR, Sharpe, max drawdown), a cumulative-return chart, risk/return statistics, drawdown path, rolling metrics, and monthly/annual heatmaps.

`reporting.performance_tearsheet(perf)` returns a `TearSheet` object whose `_repr_html_` method lets Jupyter display it inline — just make it the last expression in a cell. Native SVG `<title>` tooltips (date · value) work directly in the inline output; the richer crosshair + cursor-following label tooltip appears when the saved `.html` file is opened in a browser.


In [1]:
import math
import datetime as dt

import pandas as pd

from finstack_quant.analytics import Performance
from finstack_quant import reporting

In [2]:
# Build a deterministic 3-year daily return series using a sin-based signal.
# The seed-free sin wave makes the series fully reproducible without a random seed.
dates = pd.bdate_range("2023-01-02", periods=756, freq="B")
raw = [0.0008 + 0.012 * math.sin(i / 30.0) for i in range(len(dates))]
returns = pd.DataFrame({"Global Macro Composite": raw}, index=dates)

perf = Performance.from_returns(returns, freq="daily")
print(f"Series: {len(dates)} business days from {dates[0].date()} to {dates[-1].date()}")

Series: 756 business days from 2023-01-02 to 2025-11-24


In [3]:
# Full tear sheet — rendered inline via _repr_html_.
# Native <title> tooltips work here; open the saved .html in a browser for the
# richer crosshair + cursor-following tooltip.
reporting.performance_tearsheet(
    perf,
    title="Global Macro Composite",
    generated=dt.date(2026, 6, 19),
)

Total Return,+78.2%
Annualised Return,+22.1%
Geometric Mean,+0.1%
Mean Return,+20.2%
Annualised Volatility,13.5%
Sharpe Ratio,1.50
Sortino Ratio,2.32
Calmar Ratio,0.46
Max Drawdown,-47.7%
Skewness,-0.00
Excess Kurtosis,-1.50


In [4]:
# Trimmed variant — only summary KPIs, cumulative chart, and monthly heatmap.
reporting.performance_tearsheet(
    perf,
    sections=["summary", "cumulative", "monthly"],
    generated=dt.date(2026, 6, 19),
)

,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec,Year
2023,+11.2,+24.5,+32.4,+17.3,+0.3,-15.6,-20.6,-17.3,-2.0,+16.8,+30.3,+26.9,+125.9
2024,+12.8,-6.4,-17.9,-21.1,-13.2,+3.6,+23.4,+31.5,+22.8,+5.8,-11.4,-20.8,-9.0
2025,-19.8,-6.6,+9.8,+26.9,+30.9,+18.2,-0.2,-15.0,-21.5,-17.1,-2.8,·,-13.3


To save a standalone interactive HTML file: `ts = reporting.performance_tearsheet(perf); ts.save("tearsheet.html")`

## Takeaways

- Reporting functions are presentation wrappers over analytics, valuation, statement, or portfolio results produced earlier in the curriculum.
- Keep the analytical source of truth in the typed objects or JSON specs, then render a tear sheet for review.
- Pass fixed `generated` dates in examples so notebook output remains reproducible.
